# Functionality Overview: Extracting Garnet Data with pyMAGEMin

This notebook demonstrates the extraction and analysis of garnet (gt) compositional data along pressure–temperature (P–T) paths using the `pyMAGEMin` interface. The workflow includes both single-point and multi-point calculations, with and without fractionation, and concludes with population analysis and visualization.

---

## 1. Setup: Import Required Modules

Begin by importing core libraries and the MAGEMin interface.

In [ ]:
from pyMAGEMin.functions import MAGEMin_functions
import juliacall
import matplotlib.pyplot as plt

In [ ]:
# Load the MAGEMin Julia module
MAGEMin_C = juliacall.newmodule("MAGEMin")
MAGEMin_C.seval("using MAGEMin_C")

## 2. Define Bulk Composition and Initialize MAGEMin

Use a representative metapelite bulk composition (Shaw, 1954; Symmes & Ferry, 1991) in wt%. Specify the MAGEMin database and initialise the system.

In [ ]:
# Average pelite composition, in wt% (Shaw 1954 / Symmes & Ferry 1991)
Xoxides = ["SiO2", "Al2O3", "FeO", "MgO", "MnO", "CaO", "Na2O", "K2O", "H2O"]
X = [59.77, 16.57, 5.88, 2.62, 0.07, 2.17, 1.73, 3.53, 7.666] 
db = "mp"  # "mp": metapelite (White et al., 2014b)
sys_in = 'wt'
data = MAGEMin_C.Initialize_MAGEMin(db, verbose=False)

## 3. Single-Point Garnet Calculations

### 3.1 Endmember Fractions
Calculate garnet endmember (pyrope, almandine, spessartine, grossular, knorringite) fractions at a single P–T condition.

In [ ]:
garnet_calculator = MAGEMin_functions.MAGEMinGarnetCalculator()
gt_frac, gt_wt, gt_vol, emDict_mol, emDict_wt, out = garnet_calculator.gt_single_point_calc_endmembers(
    P=10., T=1000., data=data, X=X, Xoxides=Xoxides, sys_in=sys_in
)

### 3.2 Elemental Fractions
Calculate major element (Mg, Mn, Fe, Ca) fractions in garnet at the same conditions.

In [ ]:
gt_frac, gt_wt, gt_vol, Mg, Mn, Fe, Ca, out = garnet_calculator.gt_single_point_calc_elements(
    P=10., T=1000., data=data, X=X, Xoxides=Xoxides, sys_in=sys_in 
)

## 4. Multi-Point Garnet Calculations Along a P–T Path

### 4.1 Without Fractionation
Compute garnet composition along a P–T path, keeping the bulk composition constant.

In [ ]:
P = [5., 6., 7., 8., 9., 10.]
T = [500., 600., 700., 800., 900., 1000.]
gt_mol_frac_nf, gt_wt_frac_nf, gt_vol_frac_nf, d_gt_mol_frac_nf, d_gt_wt_frac_nf, Mgi_nf, Mni_nf, Fei_nf, Cai_nf = \
    garnet_calculator.gt_along_path(P, T, data, X, Xoxides, sys_in=sys_in, fractionate=False)

### 4.2 With Fractionation
Bulk rock composition is dynamically updated at each P–T step by subtracting the garnet fraction, simulating fractionation.

In [ ]:
gt_mol_frac, gt_wt_frac, gt_vol_frac, d_gt_mol_frac, d_gt_wt_frac, Mgi, Mni, Fei, Cai = \
    garnet_calculator.gt_along_path(P, T, data, X, Xoxides, sys_in=sys_in, fractionate=True)

## 5. Plotting Garnet Volume Fraction and Composition

Compare garnet volume fraction and element trends with and without fractionation. Solid lines = no fractionation; dashed = fractionated.

In [ ]:
plt.plot(T, gt_vol_frac_nf, c='k', label='gt vol frac (no frac)')
plt.plot(T, gt_vol_frac, ls=':', c='k', label='gt vol frac (frac)')
plt.plot(T, Mgi_nf, c='orange', label='Mg (no frac)')
plt.plot(T, Mni_nf, c='b', label='Mn (no frac)')
plt.plot(T, Cai_nf, c='g', label='Ca (no frac)')
plt.plot(T, Fei_nf, c='r', label='Fe (no frac)')
plt.plot(T, Mgi, ls=':', c='orange', label='Mg (frac)')
plt.plot(T, Mni, ls=':', c='b', label='Mn (frac)')
plt.plot(T, Cai, ls=':', c='g', label='Ca (frac)')
plt.plot(T, Fei, ls=':', c='r', label='Fe (frac)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Fraction / Composition')
plt.legend()
plt.grid()

## 6. Creating a P–T–t Path for Garnet Growth Modeling

A higher-resolution P–T–t array can be generated for more realistic growth simulations.

In [ ]:
from pyMAGEMin.functions.utils import create_PTt_path

P = [5., 6., 7., 8., 9., 10.]  # kbar
T = [500., 600., 700., 800., 900., 1000.]  # °C
t = [1., 2., 3., 4., 5., 6.]  # Myr
npoints = 15
PTt_data = create_PTt_path(P, T, t, npoints)
P_new, T_new, t_new = PTt_data[:,0], PTt_data[:,1], PTt_data[:,2]

In [ ]:
plt.plot(T_new, P_new)
plt.xlabel('Temperature (°C)')
plt.ylabel('Pressure (kbar)')

## 7. Garnet Growth with and without Fractionation Along a P–T–t Path

In [ ]:
gt_mol_frac_nf, gt_wt_frac_nf, gt_vol_frac_nf, d_gt_mol_frac_nf, d_gt_wt_frac_nf, Mgi_nf, Mni_nf, Fei_nf, Cai_nf = \
    garnet_calculator.gt_along_path(P_new, T_new, data, X, Xoxides, sys_in=sys_in, fractionate=False)

gt_mol_frac, gt_wt_frac, gt_vol_frac, d_gt_mol_frac, d_gt_wt_frac, Mgi, Mni, Fei, Cai = \
    garnet_calculator.gt_along_path(P_new, T_new, data, X, Xoxides, sys_in=sys_in, fractionate=True)

In [ ]:
plt.plot(T_new, gt_vol_frac_nf, c='k', label='gt vol frac (no frac)')
plt.plot(T_new, gt_vol_frac, ls=':', c='k', label='gt vol frac (frac)')
plt.plot(T_new, Mgi_nf, c='orange', label='Mg (no frac)')
plt.plot(T_new, Mni_nf, c='b', label='Mn (no frac)')
plt.plot(T_new, Cai_nf, c='g', label='Ca (no frac)')
plt.plot(T_new, Fei_nf, c='r', label='Fe (no frac)')
plt.plot(T_new, Mgi, ls=':', c='orange', label='Mg (frac)')
plt.plot(T_new, Mni, ls=':', c='b', label='Mn (frac)')
plt.plot(T_new, Cai, ls=':', c='g', label='Ca (frac)')
plt.plot(T_new, Fei, ls=':', c='r', label='Fe (frac)')
plt.legend()
plt.grid()

## 8. Garnet Population Generation and Analysis

Generate a synthetic population of garnets, each with its own growth history, using the `GarnetGenerator` class.

In [ ]:
from pyMAGEMin.functions.garnet_growth import GarnetGenerator

In [ ]:
from pyMAGEMin.functions.garnet_growth import GarnetGenerator

garnet_generator = GarnetGenerator(
    P_new, T_new, t_new,
    data, X, Xoxides, sys_in,
    r_min=100, r_max=1000, # microns
    garnet_classes=50,
    nR_diff=50,
    fractionate=True
)
garnet_data = garnet_generator.generate_garnets()

### 8.1 Garnet Volume Growth Summary
Visualize the normalized volume growth curve.

In [ ]:
garnet_no = 0  # Select a garnet
garnet_generator.plot_garnet_summary('N', garnet_no=garnet_no)

## 9. Extract and Plot Garnet Composition Profiles

Select the desired garnet and plot key compositional parameters as a function of radius and time.

In [ ]:
keys = ["Rr", "tr", "Pr", "Tr", "Mnr", "Mgr", "Fer", "Car"]
Rr, tr, Pr, Tr, Mnr, Mgr, Fer, Car = (garnet_data[garnet_no][k] for k in keys)
plt.plot(Rr, Mnr, label='Mn', c='blue')
plt.plot(Rr, Mgr, label='Mg', c='red')
plt.plot(Rr, Fer, label='Fe', c='magenta')
plt.plot(Rr, Car, label='Ca', c='green')
plt.legend()
plt.grid()

In [ ]:
plt.plot(tr, Mnr, label='Mn', c='blue')
plt.plot(tr, Mgr, label='Mg', c='red')
plt.plot(tr, Fer, label='Fe', c='magenta')
plt.plot(tr, Car, label='Ca', c='green')
plt.legend()
plt.grid()

In [ ]:
prograde_data = garnet_generator.get_prograde_concentrations()
tG, TG, PG, MnG, MgG, FeG, CaG = prograde_data

plt.plot(tG, MnG, label='Mn', c='blue')
plt.plot(tG, MgG, label='Mg', c='red')
plt.plot(tG, FeG, label='Fe', c='magenta')
plt.plot(tG, CaG, label='Ca', c='green')
plt.legend()
plt.grid()

## 10. Garnet Composition After Growth (Retrograde Path)

After garnet stops growing, extract and plot the retrograde compositional data.

In [ ]:
retrograde_data = garnet_generator.get_retrograde_concentrations()
tG, TG, PG, MnG, MgG, FeG, CaG = retrograde_data

In [ ]:
plt.plot(tG, MnG, label='Mn', c='blue')
plt.plot(tG, MgG, label='Mg', c='red')
plt.plot(tG, FeG, label='Fe', c='magenta')
plt.plot(tG, CaG, label='Ca', c='green')
plt.legend()
plt.grid()

In [ ]:
plt.plot(tr, Mnr, label='Mn', c='blue')
plt.plot(tr, Mgr, label='Mg', c='red')
plt.plot(tr, Fer, label='Fe', c='magenta')
plt.plot(tr, Car, label='Ca', c='green')
plt.plot([tr[-1], tr[-1]], [0, 1], c='k', ls='--')
plt.plot([tG[0], tG[0]], [0, 1], c='r', ls=':')
plt.plot(tG, MnG, label='Mn', c='blue')
plt.plot(tG, MgG, label='Mg', c='red')
plt.plot(tG, FeG, label='Fe', c='magenta')
plt.plot(tG, CaG, label='Ca', c='green')

---
### References
- Shaw., D., 1956, *Geochemistry of pelitic rocks: Part III. Major elements and general geochemistry*. Geological Society of America Bulletin, v. 67, p. 919–934
- Symmes, G. H. & Ferry, J. M., 1991, *Evidence from mineral assemblages for infiltration of pelitic schists by aqueous fluids during metamorphism*. Contr. Mineral. and Petrol. 108, 419–438

